# AI Marketing Squad

A modular multi-agent workflow for research-backed social content. The pipeline is **trend research → visual planning → copywriting → human review**.

This notebook calls the package in `src/marketing_squad` so notebook and command-line behavior stay aligned. It creates drafts only; review facts, source URLs, rights, and tone before publishing.

## 1. Install the project

Run this notebook from the repository root or from its `notebooks/` directory.

In [ ]:
from pathlib import Path
import subprocess
import sys

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists() and (ROOT.parent / "pyproject.toml").exists():
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").exists():
    raise FileNotFoundError("Open this notebook from the repository or its notebooks folder.")

subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", str(ROOT)])
print(f"Project installed from: {ROOT}")

## 2. Configure credentials

Create a local `.env` from `.env.example` and add your own OpenAI and Tavily API keys. Never commit `.env`. The notebook does not display key values.

In [ ]:
from marketing_squad.config import Settings

settings = Settings.from_env()
settings.validate_api_keys()
print(f"Configuration ready. Model: {settings.model_id}")
print(f"Carousel range: {settings.min_carousel_slides}–{settings.max_carousel_slides} slides")

## 3. Define the campaign brief

Change the topic, audience, or platforms for a different run. Trend discovery happens at execution time through the configured search tool.

In [ ]:
TOPIC = "How to explain career gaps using AI"
AUDIENCE = "early-career professionals and job seekers"
PLATFORMS = ("Instagram", "LinkedIn")

if not TOPIC.strip():
    raise ValueError("Add a campaign topic before running the workflow.")

print({"topic": TOPIC, "audience": AUDIENCE, "platforms": PLATFORMS})

## 4. Generate the draft

This call uses provider APIs and may incur charges. Output is not automatically fact-checked or published.

In [ ]:
from IPython.display import Markdown, display
from marketing_squad.squad import generate_campaign

campaign_markdown = generate_campaign(
    TOPIC,
    audience=AUDIENCE,
    platforms=PLATFORMS,
)
display(Markdown(campaign_markdown))

## 5. Review and evaluate

Verify that each research hook is supported by its linked source, that publication dates suit the claim, and that the visual plan aligns with the script. The structural evaluator works on normalized JSON and checks completeness—not truth or marketing impact.

See [`../docs/evaluation.md`](../docs/evaluation.md) for metric definitions and a suggested quality-review protocol.

In [ ]:
from marketing_squad.validation import validate_carousel_structure

# This validates the configured count only; it does not score content quality.
validation = validate_carousel_structure(
    slide_count=8,
    minimum=settings.min_carousel_slides,
    maximum=settings.max_carousel_slides,
)
print(validation.to_dict())